
#### Cell 1: Install dependencies & mount Drive


In [ ]:
!pip install -q "crewai[tools]" crewai \
    langchain langchain-openai langchain-community langchain-chroma \
    faiss-cpu \
    "unstructured[all-docs]" pypdf python-docx openpyxl \
    streamlit

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.7/67.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
%%writefile app.py
import os
import tempfile
import hashlib
import uuid
from typing import List, Tuple
import re
import streamlit as st
import streamlit.components.v1
from langchain_community.document_loaders import (
    UnstructuredExcelLoader,
    UnstructuredWordDocumentLoader,
    PyPDFLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document   # Langchain for vector store, embeddings, chunking, splitting, multimodal document loading
from crewai import Agent, LLM, Task, Crew  # Agentic Framework
from crewai_tools import SerperDevTool     # Tools and task assign becomes easy


# ---------------- CONFIG ----------------

st.set_page_config(
    page_title="SantaClaraUniversityFinChat",
    layout="wide",
)
#Shared with me > NLP > Project > FinChat
DEFAULT_KB_DIR = "/content/drive/MyDrive/FinChatBot"
LOGO_PATH = "/content/drive/MyDrive/FinChatBot/SCU_Logo.png"  # st. is alias for streamlit packages



# ---------------- GLOBAL USER STORE (PERSISTS ACROSS REFRESH) ----------------

@st.cache_resource
def get_user_store(): # User session or to store chat history.
    """
    Global in-memory store shared across reruns.
    Keys: user_hash (SHA-256 of API key)
    Values: {"sessions": {session_id: {"title": str, "messages": [...] }},
             "current_session_id": str | None}
    """
    return {}


def get_user_hash(api_key: str) -> str: # To encypted the Open API key
    """
    Hash the OpenAI API key so we never store it directly in local variables
    used for tracking the user.
    """
    return hashlib.sha256(api_key.encode("utf-8")).hexdigest()


# ---------------- HELPERS ----------------

def get_openai_api_key() -> str:
    """Get OpenAI API key from env or sidebar."""
    if "OPENAI_API_KEY" in os.environ and os.environ["OPENAI_API_KEY"].strip():
        return os.environ["OPENAI_API_KEY"].strip()


    api_key = st.sidebar.text_input(
     "Enter Your OpenAI API Key",
        type="password",
        help="Your key is only used locally in this app.",
    )
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key
    return api_key


def get_serper_api_key() -> str: # You can get it from Serper's site, which belongs to google and easier for websearch
    """Get Serper API key for web search (optional but recommended)."""
    if "SERPER_API_KEY" in os.environ and os.environ["SERPER_API_KEY"].strip():
        return os.environ["SERPER_API_KEY"].strip()

    serper_key = st.sidebar.text_input(
        "Serper API Key (for web search)",
        type="password",
        help="Get free API key from serper.dev for web search capability.",
    )
    if serper_key:
        os.environ["SERPER_API_KEY"] = serper_key
    return serper_key


def is_scu_related_question(question: str) -> bool:
    """
    Determine if a question is related to Santa Clara University.
    Returns True if SCU-related, False otherwise.
    """
    # Keywords that indicate SCU-related questions
    scu_keywords = [
        'scu', 'santa clara', 'leavey', 'msfa', 'finance department',
        'finance major', 'finance program', 'finance course', 'fnce',
        'real estate minor', 'silicon valley', 'bronco', 'mission',
        'admission', 'requirement', 'prerequisite', 'curriculum',
        'faculty', 'professor', 'degree', 'graduate program',
        'undergraduate', 'tuition', 'scholarship', 'deadline', 'financial aid'
    ]

    question_lower = question.lower()

    # Check if any SCU keyword is in the question
    for keyword in scu_keywords:
        if keyword in question_lower:
            return True

    return False


def make_text_splitter() -> RecursiveCharacterTextSplitter:
    return RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
        separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""],
    )


def load_docs_from_directory(directory: str) -> List[Document]:
    docs: List[Document] = []

    if not os.path.isdir(directory):
        st.warning(f"Directory '{directory}' does not exist yet.")
        return docs

    for root, _, files in os.walk(directory):
        for filename in files:
            ext = os.path.splitext(filename)[1].lower()
            file_path = os.path.join(root, filename)

            try:
                if ext == ".pdf":
                    loader = PyPDFLoader(file_path)
                elif ext in (".docx", ".doc"):
                    loader = UnstructuredWordDocumentLoader(file_path)
                elif ext in (".xlsx", ".xls"):
                    loader = UnstructuredExcelLoader(file_path)
                else:
                    continue

                file_docs = loader.load()
                for d in file_docs:
                    d.metadata.setdefault("source", file_path)
                docs.extend(file_docs)
            except Exception as e:
                st.error(f"Error loading {file_path}: {e}")

    return docs


def load_docs_from_uploaded_files(uploaded_files) -> List[Document]:
    docs: List[Document] = []
    if not uploaded_files:
        return docs

    for uploaded_file in uploaded_files:
        ext = os.path.splitext(uploaded_file.name)[1].lower()

        with tempfile.NamedTemporaryFile(delete=False, suffix=ext) as tmp:
            tmp.write(uploaded_file.read())
            tmp_path = tmp.name

        try:
            if ext == ".pdf":
                loader = PyPDFLoader(tmp_path)
            elif ext in (".docx", ".doc"):
                loader = UnstructuredWordDocumentLoader(tmp_path)
            elif ext in (".xlsx", ".xls"):
                loader = UnstructuredExcelLoader(tmp_path)
            else:
                st.warning(f"Unsupported file type: {uploaded_file.name}")
                continue

            file_docs = loader.load()
            for d in file_docs:
                d.metadata.setdefault("source", uploaded_file.name)
            docs.extend(file_docs)
        except Exception as e:
            st.error(f"Error loading {uploaded_file.name}: {e}")

    return docs

#2 vectorbases (chroma db first then faiss)
VectorStoreTuple = Tuple[Chroma, FAISS]

def build_vectorstore(docs: List[Document], api_key: str, persist_path: str) -> VectorStoreTuple:
    if not docs:
        raise ValueError("No documents to index.")

    splitter = make_text_splitter()
    split_docs = splitter.split_documents(docs)
    embeddings = OpenAIEmbeddings(openai_api_key=api_key)

    # --- 1. CHROMA (Primary & Persistent) ---
    chroma_path = persist_path + "/chroma_db"

    chroma_vectorstore = Chroma.from_documents(
        documents=split_docs,
        embedding=embeddings,
        persist_directory=chroma_path
    )
    chroma_vectorstore.persist()

    # --- 2. FAISS (Secondary & In-Session) ---
    # FAISS is fast and good for a secondary index. We build it in-memory.
    faiss_vectorstore = FAISS.from_documents(
        documents=split_docs,
        embedding=embeddings
    )

    return chroma_vectorstore, faiss_vectorstore


# ---------- RETRIEVAL FROM LOCAL SOURCES (KB + UPLOADS) ----------

def retrieve_context(
    question: str,
    kb_chroma_vs: Chroma | None,         # Chroma KB store
    kb_faiss_vs: FAISS | None,           # FAISS KB store
    upload_chroma_vs: Chroma | None,     # Chroma Upload store
    upload_faiss_vs: FAISS | None,       # FAISS Upload store
) -> Tuple[str, List[Document]]:
    """Retrieve from KB + uploads using both Chroma and FAISS. Returns context string + docs"""
    all_docs: List[Document] = []

    # --- 1. SEARCH KB (CHROMA then FAISS) ---
    if kb_chroma_vs is not None:
        # Search Chroma first (the persistent store)
        all_docs.extend(kb_chroma_vs.similarity_search(question, k=4))
    if kb_faiss_vs is not None:
        # Search FAISS second (the in-memory store)
        all_docs.extend(kb_faiss_vs.similarity_search(question, k=4))

    # --- 2. SEARCH UPLOADS (CHROMA then FAISS) ---
    if upload_chroma_vs is not None:
        all_docs.extend(upload_chroma_vs.similarity_search(question, k=4))
    if upload_faiss_vs is not None:
        all_docs.extend(upload_faiss_vs.similarity_search(question, k=4))

    # Deduplicate by (page_content, source)
    seen = set()
    unique_docs: List[Document] = []
    for d in all_docs:
        key = (d.page_content.strip(), d.metadata.get("source"))
        if key not in seen:
            seen.add(key)
            unique_docs.append(d)

    # Build context string with indices
    context_chunks = []
    for idx, doc in enumerate(unique_docs):
        src = doc.metadata.get("source", "unknown")
        context_chunks.append(
            f"[{idx+1}] Source: {src}\n{doc.page_content}"
        )
    context_text = "\n\n".join(context_chunks) if context_chunks else "[no retrieved context from local documents]"
    return context_text, unique_docs


# ---------- CREWAI AGENT WITH WEB SEARCH TOOL ----------

def get_crew_agent_and_tool(use_web: bool):
    """Create or reuse a CrewAI agent with optional web search capability."""

    llm = LLM(model="gpt-4o-mini")

    # Initialize web search tool if requested and API key is available
    tools = []
    if use_web and os.environ.get("SERPER_API_KEY"):
        search_tool = SerperDevTool()
        tools.append(search_tool)

    agent = Agent(
        role="Finance Dept RAG Assistant",
            goal=(
                "Answer user questions accurately using the provided context from the "
                "knowledge base, uploaded documents, and web search results. "
                "Be professional, friendly, honest, and helpful, and make any web "
                "links clickable with Markdown."
            ),
            backstory=(
                # --- START: ADDED GENERAL CONTEXT ---
                "The Finance Department at Santa Clara University’s Leavey School of Business "
                "offers undergraduate and graduate programs that blend foundational finance "
                "education with real-world analytical skills. Undergraduate students can pursue "
                "a Finance Major, which emphasizes quantitative methods, ethics, financial "
                "decision-making, and practical application, or a Real Estate Minor, which "
                "prepares students to address Silicon Valley’s commercial and residential real "
                "estate challenges. Graduate students can pursue the MS in Finance & Analytics "
                "(MSFA), an industry-connected program combining corporate finance, investment "
                "management, and capital markets with advanced analytics tools such as R, "
                "Python, and SQL, available both on-campus and online. The department’s faculty "
                "are highly published scholars and industry practitioners, and its Silicon "
                "Valley location provides access to internships and career opportunities across "
                "tech firms, banks, VC firms, consulting, and financial institutions.\\n\\n"
                # --- END: ADDED GENERAL CONTEXT ---

               "You are an academic advisor-focused AI assistant that supports users with "
                "questions about Santa Clara University's Finance Department programs, courses, requirements.\\n\\n"

                "You always receive:\n"
                "- A user question.\n"
                "- A context block that may include:\n"
                "  * Pre-loaded finance knowledge-base documents.\n"
                "  * User-uploaded documents (e.g., checklists, PDFs, notes).\n"
                "  * Web search results with titles and URLs.\n\n"

                "Core behavior:\n"
                "- Prioritize the provided context when answering. If context and your "
                "general knowledge disagree, assume the context is correct.\n"
                "- Never invent requirements, dates, course names, or URLs. If the "
                "information is not present or is ambiguous in the context, say so "
                "clearly and explain what is missing.\n"
                "- When you rely on specific documents, refer to them naturally, e.g. "
                "'According to the MSFA checklist...' or 'In the finance bulletin...'.\n"
                "- You may use your general knowledge to clarify concepts, but you must "
                "distinguish clearly between what comes from the provided context and "
                "what is general background knowledge.\n\n"

                # --- START: DRAHMANN CENTER FALLBACK LOGIC ---
                "If you cannot answer a question due to missing, insufficient, or unclear "
                "information in the provided context, you must:\n"
                "1. State clearly that the available context does not provide the required answer.\n"
                "2. Recommend that the student contact the Drahmann Academic Advising Center.\n\n"
                "When referring to the Drahmann Center, provide this exact information:\n"
                "- **Location:** Kenna Hall 101\n"
                "- **Phone:** (408) 554-4318\n"
                "- **Email:** drahmanncenter@scu.edu\n"
                "- **Hours:** Monday–Friday, 9:00am–4:00pm PDT\n"
                "- The Drahmann Center is the home for undergraduate academic advising at SCU "
                "and supports Core Curriculum guidance, major/minor planning, academic policies, "
                "registration questions, and academic learning resources.\n\n"
                "Use this fallback only when necessary—after attempting to answer using "
                "the provided documents, knowledge base, and (if enabled) web search.\n\n"
                # --- END: DRAHMANN CENTER FALLBACK LOGIC ---

               "Tone and style:\\n"
                "- Be professional but warm and friendly.\\n"
                "- Be concise and well-structured, using headings and bullet points \\n"
                "when they help readability.\\n"
                "- Be honest about uncertainty or gaps; never pretend to know something \\n"
                "you do not.\\n"

                # --- START: GUARDRAIL ADDITION ---
                "- You do NOT provide personalized investment advice. You focus on \\n"
                "explaining academic information.\\n\\n"

                "Guardrails:\\n"
                "- You **must refuse** to answer any questions that are harmful, dangerous, \\n"
                "illegal, or promote self-harm, violence, or discrimination.\\n"
                "- If a question touches upon these sensitive topics, respond with a polite \\n"
                "refusal, state that your purpose is academic assistance only, and recommend \\n"
                "seeking help from appropriate professional resources.\\n"
                "- Never provide personalized financial or trading recommendations.\\n"
                # --- END: GUARDRAIL ADDITION ---
            ),
        llm=llm,
        tools=tools,
        verbose=True,
        allow_delegation=False,
    )

    return agent


def run_agent_answer(question: str, context_text: str, use_web: bool, is_scu_related: bool) -> str:
    """
    Use the CrewAI agent to generate the final answer.
    The agent will intelligently route between local context and web search.
    """
    agent = get_crew_agent_and_tool(use_web)

    # Build the prompt based on question type
    if is_scu_related:
        search_guidance = """
   - **IMPORTANT for SCU questions**: If local context is insufficient, search using
     "site:scu.edu" to prioritize official SCU sources
   - Example: "MSFA program requirements site:scu.edu" or "Santa Clara University Finance admission site:scu.edu"
"""
        drahmann_guidance = """
**If you cannot answer the student's SCU academic question after using the local context
and (if enabled) web search**, you should:

- Clearly state that the available information is insufficient, and
- Recommend that the student contact the **Drahmann Academic Advising Center** for official guidance.

Provide this exact information:
- **Location:** Kenna Hall 101
- **Phone:** (408) 554-4318
- **Email:** drahmanncenter@scu.edu
- **Hours:** Monday–Friday, 9:00am–4:00pm PDT

Explain that the Drahmann Center supports undergraduate advising, Core Curriculum questions,
and major/minor planning.
"""
    else:
        search_guidance = """
   - **IMPORTANT for general questions**: Use web search directly to find current, reliable information
   - Search broadly for the most authoritative sources on the topic
"""
        drahmann_guidance = ""

    prompt = f"""
You are a versatile knowledge assistant that can answer both SCU Finance Department related questions and general knowledge questions.

User question:
\"\"\"{question}\"\"\"

Question type: {"SCU-related" if is_scu_related else "General knowledge"}

Retrieved context from local documents:
\"\"\"{context_text}\"\"\"

Your task:
1. Carefully read the user question
2. Determine if you need to use web search:
   - For SCU questions: Use local context first; if insufficient, search site:scu.edu
   - For general questions: Use web search to get current information
3. Answer accurately and cite all sources

{search_guidance}

**Source Citation Rules**:
- For web sources: Format as clickable Markdown links [Title](URL)
- Always include a "Sources" or "References" section at the end with all web URLs used
- For local documents: Cite naturally (e.g., "According to the MSFA handbook...")
- If you use web search, you MUST list the source URLs you relied on

**Response Structure**:
1. Direct answer to the question
2. Supporting details/explanation
3. **Sources** section (if web search was used) with all clickable links:
   - [Source Title 1](URL1)
   - [Source Title 2](URL2)

**Important Guidelines**:
- Be accurate and honest - don't make up information
- If you can't find sufficient information, say so clearly
- For SCU questions: Prefer official scu.edu sources over third-party sites
- For general questions: Prefer authoritative, well-known sources
- Never provide personalized financial/investment advice
- Refuse harmful, dangerous, or illegal requests politely

{drahmann_guidance}

Now provide your best answer following all instructions above.
"""

    # Create a task for the agent
    task = Task(
        description=prompt,
        agent=agent,
        expected_output="A comprehensive answer with proper source citations and clickable links for web sources."
    )

    # Create and run the crew
    crew = Crew(
        agents=[agent],
        tasks=[task],
        verbose=False
    )

    result = crew.kickoff()

    # Extract the answer from the result
    answer = getattr(result, "raw", str(result))
    return answer


def extract_urls_from_answer(answer: str) -> List[Tuple[str, str]]:
    """
    Extract URLs from markdown links in the answer.
    Returns list of (title, url) tuples.
    """
    # Pattern to match markdown links: [text](url)
    pattern = r'\[([^\]]+)\]\(([^\)]+)\)'
    matches = re.findall(pattern, answer)

    # Also look for standalone URLs
    url_pattern = r'https?://[^\s\)]+'
    standalone_urls = re.findall(url_pattern, answer)

    urls = []
    seen = set()

    # Add markdown links
    for title, url in matches:
        if url not in seen:
            urls.append((title, url))
            seen.add(url)

    # Add standalone URLs
    for url in standalone_urls:
        if url not in seen:
            urls.append((url, url))
            seen.add(url)

    return urls

# ---------------------------------- Voice Chat -------------------------------------------------------
def add_voice_input_to_chat():
    """
    Returns HTML/JS code that adds voice recognition to the Streamlit chat input.
    The mic button appears inside the chat input box with visual feedback.
    """
    return """
    <script>
    (function() {
        // Wait for Streamlit to load
        function initVoiceInput() {
            const chatInput = window.parent.document.querySelector('input[aria-label="💭 Ask anything - SCU Finance or general knowledge..."]') ||
                            window.parent.document.querySelector('textarea[aria-label="💭 Ask anything - SCU Finance or general knowledge..."]') ||
                            window.parent.document.querySelector('.stChatInputContainer input') ||
                            window.parent.document.querySelector('.stChatInputContainer textarea');

            if (!chatInput) {
                setTimeout(initVoiceInput, 500);
                return;
            }

            // Check if already initialized
            if (chatInput.dataset.voiceInitialized) return;
            chatInput.dataset.voiceInitialized = 'true';

            // Create mic button
            const micButton = window.parent.document.createElement('button');
            micButton.innerHTML = '🗣️';
            micButton.style.cssText = `
                position: absolute;
                right: 50px;
                top: 50%;
                transform: translateY(-50%);
                background: transparent;
                border: none;
                font-size: 24px;
                cursor: pointer;
                padding: 8px;
                z-index: 1000;
                transition: all 0.3s ease;
                border-radius: 50%;
            `;
            micButton.title = 'Hold to speak';

            // Create status indicator
            const statusDiv = window.parent.document.createElement('div');
            statusDiv.style.cssText = `
                position: absolute;
                right: 90px;
                top: 50%;
                transform: translateY(-50%);
                background: #ff4444;
                color: white;
                padding: 4px 12px;
                border-radius: 12px;
                font-size: 12px;
                font-weight: bold;
                display: none;
                z-index: 999;
                animation: pulse 1.5s infinite;
            `;
            statusDiv.textContent = '🔴 Listening...';

            // Add pulse animation
            const style = window.parent.document.createElement('style');
            style.textContent = `
                @keyframes pulse {
                    0%, 100% { opacity: 1; }
                    50% { opacity: 0.7; }
                }
            `;
            window.parent.document.head.appendChild(style);

            // Position container
            const inputContainer = chatInput.closest('.stChatInputContainer') || chatInput.parentElement;
            if (inputContainer) {
                inputContainer.style.position = 'relative';
                inputContainer.appendChild(micButton);
                inputContainer.appendChild(statusDiv);
            }

            // Speech Recognition setup
            const SpeechRecognition = window.SpeechRecognition || window.webkitSpeechRecognition;

            if (!SpeechRecognition) {
                micButton.title = 'Speech recognition not supported in this browser';
                micButton.style.opacity = '0.3';
                micButton.style.cursor = 'not-allowed';
                return;
            }

            const recognition = new SpeechRecognition();
            recognition.continuous = true;
            recognition.interimResults = true;
            recognition.lang = 'en-US';

            let isListening = false;
            let finalTranscript = '';
            let interimTranscript = '';

            // Mouse/Touch events for hold-to-speak
            const startListening = (e) => {
                e.preventDefault();
                if (isListening) return;

                isListening = true;
                finalTranscript = '';
                interimTranscript = '';

                // Visual feedback
                micButton.style.background = '#ff4444';
                micButton.style.transform = 'translateY(-50%) scale(1.1)';
                statusDiv.style.display = 'block';

                try {
                    recognition.start();
                } catch (e) {
                    console.log('Recognition already started');
                }
            };

            const stopListening = (e) => {
                e.preventDefault();
                if (!isListening) return;

                isListening = false;

                // Visual feedback
                micButton.style.background = 'transparent';
                micButton.style.transform = 'translateY(-50%) scale(1)';
                statusDiv.style.display = 'none';

                recognition.stop();

                // Set the input value and trigger all necessary events for Streamlit
                if (finalTranscript.trim()) {
                    const transcript = finalTranscript.trim();

                    // First, set the value normally
                    chatInput.value = transcript;

                    // Then use the native setter trick for React
                    try {
                        const nativeInputValueSetter = Object.getOwnPropertyDescriptor(
                            window.HTMLInputElement.prototype, 'value'
                        ).set;
                        nativeInputValueSetter.call(chatInput, transcript);
                    } catch (e) {
                        console.log('Native setter not available, using fallback');
                    }

                    // Focus first
                    chatInput.focus();

                    // Trigger input event (most important for Streamlit)
                    const inputEvent = new Event('input', { bubbles: true, cancelable: true });
                    chatInput.dispatchEvent(inputEvent);

                    // Small delay then trigger change event
                    setTimeout(() => {
                        const changeEvent = new Event('change', { bubbles: true, cancelable: true });
                        chatInput.dispatchEvent(changeEvent);

                        // Position cursor at end
                        chatInput.setSelectionRange(transcript.length, transcript.length);

                        // Trigger a fake key event to wake up Streamlit
                        const keyEvent = new KeyboardEvent('keyup', {
                            bubbles: true,
                            cancelable: true,
                            key: ' ',
                            code: 'Space'
                        });
                        chatInput.dispatchEvent(keyEvent);
                    }, 50);
                }
            };

            // Event listeners for hold-to-speak
            micButton.addEventListener('mousedown', startListening);
            micButton.addEventListener('mouseup', stopListening);
            micButton.addEventListener('mouseleave', stopListening);
            micButton.addEventListener('touchstart', startListening);
            micButton.addEventListener('touchend', stopListening);

            // Ensure Enter key works after voice input
            chatInput.addEventListener('keydown', (e) => {
                if (e.key === 'Enter' && !e.shiftKey) {
                    // Ensure the value is properly registered
                    const currentValue = chatInput.value;
                    if (currentValue.trim()) {
                        // Re-trigger input event to ensure Streamlit has the latest value
                        const inputEvent = new Event('input', { bubbles: true, cancelable: true });
                        chatInput.dispatchEvent(inputEvent);
                    }
                }
            });

            // Recognition result handling
            recognition.onresult = (event) => {
                interimTranscript = '';

                for (let i = event.resultIndex; i < event.results.length; i++) {
                    const transcript = event.results[i][0].transcript;
                    if (event.results[i].isFinal) {
                        finalTranscript += transcript + ' ';
                    } else {
                        interimTranscript += transcript;
                    }
                }

                // Show interim results in input - simple and direct
                chatInput.value = (finalTranscript + interimTranscript).trim();
            };

            recognition.onerror = (event) => {
                console.error('Speech recognition error:', event.error);
                isListening = false;
                micButton.style.background = 'transparent';
                micButton.style.transform = 'translateY(-50%) scale(1)';
                statusDiv.style.display = 'none';

                if (event.error === 'not-allowed') {
                    alert('Microphone access denied. Please allow microphone access in your browser settings.');
                }
            };

            recognition.onend = () => {
                if (isListening) {
                    try {
                        recognition.start();
                    } catch (e) {
                        console.log('Recognition restart failed');
                    }
                }
            };

            // Hover effect
            micButton.addEventListener('mouseenter', () => {
                if (!isListening) {
                    micButton.style.background = 'rgba(255, 68, 68, 0.1)';
                }
            });

            micButton.addEventListener('mouseleave', () => {
                if (!isListening) {
                    micButton.style.background = 'transparent';
                }
            });
        }

        // Initialize when DOM is ready
        if (window.parent.document.readyState === 'loading') {
            window.parent.document.addEventListener('DOMContentLoaded', initVoiceInput);
        } else {
            initVoiceInput();
        }

        // Re-initialize on Streamlit reruns
        setInterval(initVoiceInput, 1000);
    })();
    </script>
    """


# ---------------- STREAMLIT UI ----------------

st.title("🎓 Welcome to SCU's Finance Department ChatBot!")

st.markdown("Ask me anything about **Santa Clara University Finance programs** or **general knowledge questions**!")

# Add voice recognition integrated with chat input (hidden component)
st.components.v1.html(add_voice_input_to_chat(), height=0)

api_key = get_openai_api_key()
if not api_key:
    st.stop()

serper_key = get_serper_api_key()


# ---------- USER IDENTIFICATION (HASHED API KEY) ----------

user_hash = get_user_hash(api_key)
user_store = get_user_store()

if user_hash not in user_store:
    user_store[user_hash] = {
        "sessions": {},
        "current_session_id": None, # hash to encrypt API key
    }

user_data = user_store[user_hash]

# ---------- SESSION STATE INITIALIZATION ----------

if "sessions" not in st.session_state:
    st.session_state.sessions = user_data["sessions"]
    st.session_state.current_session_id = user_data["current_session_id"]

# Ensure at least one session exists
if not st.session_state.sessions:
    first_session_id = uuid.uuid4().hex
    st.session_state.sessions[first_session_id] = {
        "title": "New chat",
        "messages": [],
    }
    st.session_state.current_session_id = first_session_id
    user_data["sessions"] = st.session_state.sessions
    user_data["current_session_id"] = first_session_id

if "kb_chroma_vs" not in st.session_state:
    st.session_state.kb_chroma_vs = None
if "kb_faiss_vs" not in st.session_state:
    st.session_state.kb_faiss_vs = None
if "upload_chroma_vs" not in st.session_state:
    st.session_state.upload_chroma_vs = None
if "upload_faiss_vs" not in st.session_state:
    st.session_state.upload_faiss_vs = None

# Initialize session to delete
if "session_to_delete" not in st.session_state:
    st.session_state.session_to_delete = None


# -------- Sidebar --------

# Small logo at top-left of sidebar
try:
    st.sidebar.image(LOGO_PATH, width=100)
except Exception:
    pass

st.sidebar.header(" Knowledge Base")

kb_dir = st.sidebar.text_input(
    "KB folder path",
    value=DEFAULT_KB_DIR,
    help="Put your finance PDFs/Word/Excel files here.",
)

if st.sidebar.button("🔄 Build / Rebuild KB Index"):
    with st.spinner("Loading and indexing KB documents (Chroma & FAISS)..."):
        kb_docs = load_docs_from_directory(kb_dir)
        if kb_docs:
            # Build both Chroma (persistent) and FAISS (in-memory)
            chroma_vs, faiss_vs = build_vectorstore(kb_docs, api_key, kb_dir)
            st.session_state.kb_chroma_vs = chroma_vs
            st.session_state.kb_faiss_vs = faiss_vs

            st.sidebar.success(
                f"✅ KB index built (Chroma saved to disk, FAISS in memory) "
                f"from {len(kb_docs)} documents!"
            )
        else:
            st.sidebar.warning("⚠️ No documents found in the KB folder.")

st.sidebar.markdown("---")
st.sidebar.header("📤 Runtime Uploads")

uploaded_files = st.sidebar.file_uploader(
    "Upload PDF / Word / Excel files",
    type=["pdf", "docx", "doc", "xlsx", "xls"],
    accept_multiple_files=True,
)

if st.sidebar.button("📦 Index Uploaded Files"):
    if not uploaded_files:
        st.sidebar.warning(" Please upload at least one file first.")
    else:
        # Use a temporary directory for uploaded file persistence (Chroma)
        upload_persist_dir = os.path.join(tempfile.gettempdir(), "uploaded_chroma_db")
        os.makedirs(upload_persist_dir, exist_ok=True)

        with st.spinner("Loading and indexing uploaded documents (Chroma & FAISS)..."):
            upload_docs = load_docs_from_uploaded_files(uploaded_files)
            if upload_docs:
                chroma_vs, faiss_vs = build_vectorstore(
                    upload_docs,
                    api_key,
                    upload_persist_dir,
                )
                st.session_state.upload_chroma_vs = chroma_vs
                st.session_state.upload_faiss_vs = faiss_vs

                st.sidebar.success(
                    f"✅ Indexed {len(upload_docs)} uploaded documents!"
                )
            else:
                st.sidebar.warning("⚠️ No documents could be loaded from uploads.")

st.sidebar.markdown("---")

# Web search toggle
use_web = st.sidebar.checkbox(
    "🌐 Enable Web Search",
    value=True if serper_key else False,
    help="Allows answering general questions and finding current information.",
)

if use_web and not serper_key:
    st.sidebar.warning(" Web search enabled but no Serper API key provided. Get one free at [serper.dev](https://serper.dev)")

# -------- Sidebar: Chat Sessions --------
st.sidebar.markdown("---")
st.sidebar.header("💬 Chat Sessions")

col1, col2 = st.sidebar.columns([3, 1])
with col1:
    if st.button("➕ New Chat", use_container_width=True):
        new_session_id = uuid.uuid4().hex
        st.session_state.sessions[new_session_id] = {
            "title": "Current chat",
            "messages": [],
        }
        st.session_state.current_session_id = new_session_id
        user_data["sessions"] = st.session_state.sessions
        user_data["current_session_id"] = new_session_id
        st.rerun()

# Show existing sessions
for sid, session in list(st.session_state.sessions.items()):
    title = session.get("title", "Current Chat")
    is_current = sid == st.session_state.current_session_id

    col1, col2 = st.sidebar.columns([4, 1])

    with col1:
        label = f"{title}" if is_current else f"⚪ {title}"
        if st.button(label, key=f"session_btn_{sid}", use_container_width=True):
            st.session_state.current_session_id = sid
            user_data["current_session_id"] = sid
            st.rerun()

    with col2:
        if st.button("🗑️", key=f"delete_btn_{sid}", help="Delete this chat"):
            st.session_state.session_to_delete = sid
            st.rerun()

# Handle deletion confirmation
if st.session_state.session_to_delete:
    sid_to_delete = st.session_state.session_to_delete

    st.sidebar.markdown("---")
    st.sidebar.warning(f"⚠️ Delete chat: '{st.session_state.sessions[sid_to_delete]['title']}'?")

    col1, col2 = st.sidebar.columns(2)
    with col1:
        if st.button("✅ Yes, Delete", use_container_width=True):
            # Delete the session
            del st.session_state.sessions[sid_to_delete]
            user_data["sessions"] = st.session_state.sessions

            # If we deleted the current session, switch to another or create new
            if st.session_state.current_session_id == sid_to_delete:
                if st.session_state.sessions:
                    st.session_state.current_session_id = list(st.session_state.sessions.keys())[0]
                else:
                    # Create a new session if all were deleted
                    new_session_id = uuid.uuid4().hex
                    st.session_state.sessions[new_session_id] = {
                        "title": "New chat",
                        "messages": [],
                    }
                    st.session_state.current_session_id = new_session_id

                user_data["current_session_id"] = st.session_state.current_session_id
                user_data["sessions"] = st.session_state.sessions

            st.session_state.session_to_delete = None
            st.rerun()

    with col2:
        if st.button("❌ Cancel", use_container_width=True):
            st.session_state.session_to_delete = None
            st.rerun()


# -------- Main Chat Area --------

current_session_id = st.session_state.current_session_id
current_session = st.session_state.sessions.get(
    current_session_id, {"title": "New chat", "messages": []}
)
messages = current_session["messages"]

# Show previous messages
for msg in messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Input
user_query = st.chat_input("💭 Ask anything - SCU Finance Departmnt or general knowledge...")

if user_query:
    # Add user message
    messages.append({"role": "user", "content": user_query})

    # Update session title from first message
    if len(messages) == 1:
        short_title = user_query.strip()
        if len(short_title) > 30:
            short_title = short_title[:30] + "..."
        st.session_state.sessions[current_session_id]["title"] = short_title or "Current chat"
        user_data["sessions"] = st.session_state.sessions

    with st.chat_message("user"):
        st.markdown(user_query)

    # Generate answer
    with st.chat_message("assistant"):
        used_docs: List[Document] = []

        # Determine if question is SCU-related
        is_scu_question = is_scu_related_question(user_query)

        # Show indicator of question type
        if is_scu_question:
            status_msg = st.empty()
            status_msg.info(" Detected SCU-related question. Searching knowledge base...")
        else:
            status_msg = st.empty()
            if use_web:
                status_msg.info(" General question detected. Searching the web...")
            else:
                status_msg.info(" Processing your question...")

        with st.spinner("Thinking with CrewAI..."):
            try:
                # 1) Retrieve context from local sources (only if SCU-related)
                if is_scu_question:
                  context_text, used_docs = retrieve_context(
                    user_query,
                    st.session_state.kb_chroma_vs,
                    st.session_state.kb_faiss_vs,
                    st.session_state.upload_chroma_vs,
                    st.session_state.upload_faiss_vs,
                  )
                else:
                  context_text = "[This is a general knowledge question – no local context needed]"
                  used_docs = []


                # 2) Let CrewAI agent synthesize the answer
                answer = run_agent_answer(user_query, context_text, use_web, is_scu_question)

                status_msg.empty()  # Clear the status message

            except Exception as e:
                status_msg.empty()
                st.error(f"Error generating answer: {e}")
                answer = "Sorry, something went wrong while answering your question. Please try again."
                used_docs = []

        # Main answer
        st.markdown(answer)
        messages.append({"role": "assistant", "content": answer})

        # Persist updated messages
        st.session_state.sessions[current_session_id]["messages"] = messages
        user_data["sessions"] = st.session_state.sessions

        # Extract and display web sources in a prominent box
        web_sources = extract_urls_from_answer(answer)
        if web_sources:
            st.markdown("---")
            st.markdown("### 🔗 Web Sources Used")
            for title, url in web_sources:
                # Display as clickable links
                st.markdown(f"- [{title}]({url})")

        # Show retrieved local sources in expandable panel (only for SCU questions)
        if used_docs and is_scu_question:
            st.markdown("---")
            with st.expander(" View Local Knowledge Base Sources"):
                for i, d in enumerate(used_docs, start=1):
                    src = d.metadata.get("source", "unknown")
                    st.markdown(f"**[{i}] Source:** `{src}`")
                    st.text(
                        d.page_content[:800]
                        + ("..." if len(d.page_content) > 800 else "")
                    )
                    st.markdown("---")

Writing app.py


In [ ]:
!pip install pyngrok

In [ ]:
!pkill ngrok

In [ ]:

!killall ngrok
!pkill -f ngrok


ngrok: no process found


In [ ]:
from pyngrok import ngrok


ngrok.set_auth_token("token")

In [ ]:
import subprocess
import time
import os

# Kill any existing Streamlit processes (ignore errors)
try:
    subprocess.run(["pkill", "-f", "streamlit"], check=False)
except Exception:
    pass

# Start Streamlit in the background
streamlit_cmd = [
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    "8501",
    "--server.address",
    "0.0.0.0",
    "--server.headless",
    "true",
    "--server.enableXsrfProtection",
    "false",
    "--server.enableCORS",
    "false",
]

process = subprocess.Popen(streamlit_cmd)

# Give Streamlit a moment to start
time.sleep(5)

In [ ]:
public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://annalise-creolized-loathly.ngrok-free.dev" -> "http://localhost:8501">